# Reasoning, RAG and Multi-Agent Setup (ReAct Agents)

In [1]:
import os
import yfinance as yf
from dotenv import load_dotenv
from ddgs import DDGS
from openai import OpenAI
import lancedb
from lancedb.embeddings import get_registry
import PyPDF2
import pandas as pd

In [15]:
# 1. Loading environment variables
try:
    load_dotenv()
except Exception as e:
    print(f"Error loading env variables: {e}")

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))


# 2. Downloading and extracting PDF text from path
def extract_text_from_pdf(file="nvidia_report.pdf"):
    try:
        print(" Downloading PDF...")
        reader = PyPDF2.PdfReader(file)
        file_content = ''
        for page in reader.pages:
            file_content += " " + page.extract_text()

        return file_content.strip()
    except Exception as e:
        print(f"Erro download or extrating text from PDF: {e}")
        return "" 


# 3. Creating Vetorial Database
db_uri = "rag/lancedb"
os.makedirs(db_uri, exist_ok=True)

try:
    print("Starting vectorial database LanceDB...")
    db = lancedb.connect(db_uri)

    if "vectordb" in db.list_tables():
        table = db.open_table("vectordb")
        print("Existent vectorial database loaded.")
    else:
        # Download and process the PDF
        # url_pdf = "https://s201.q4cdn.com/141608511/files/doc_financials/2024/ar/NVIDIA-2024-Annual-Report.pdf"
        pdf_text = extract_text_from_pdf()

        # pdf_text already extracted and generating blocks:
        blocks = [pdf_text[i:i+2000] for i in range(0, len(pdf_text), 2000)]

        print("Generating embeddings using the OpenAI api...")

        # 1) Call embeddings
        try:
            emb_response = client.embeddings.create(
                model="text-embedding-3-small",
                input=blocks  # lista de strings
            )
            # 2) Extracting embeddings
            embeddings = [item.embedding for item in emb_response.data]

            # 3) Creating the dataframe
            data = pd.DataFrame({
                "id": range(len(blocks)),
                "text": blocks,
                "embedding": embeddings
            })

            # 4) Creating a table on LanceDB (Lance will allow 'embedding' column as vector)
            table = db.create_table("vectordb", data=data)
            print("Vector database successfuly created.")

        except Exception as e:
            print("Erro generating OpenAI embeddings:", e)
except Exception as e:
    print(f"Erro creating or loading vector database: {e}")

Starting vectorial database LanceDB...
Generating embeddings using the OpenAI api...
Vector database successfuly created.


In [16]:
# 4. Agents
class SearchAgent:
    def search(self, query):
        try:
            with DDGS() as ddgs:
                results = list(ddgs.text(query, max_results=3))
            return results
        except Exception as e:
            return [f"Search error: {e}"]


class FinancialAgent:
    def __init__(self, ticker="NVDA"):
        self.ticker = ticker

    def collect_data(self):
        try:
            adction = yf.Ticker(self.ticker)
            data = {
                "company": adction.info.get("longName"),
                "price": adction.history(period="1d")["Close"].iloc[-1],
                "sector": adction.info.get("sector"),
            }
            return data
        except Exception as e:
            return {"error": str(e)}

In [17]:
# 5. RAG function (vectorial search)
def rag_search(query, table, client, top_k=3):
    try:
        print("Generating embedding from query and searching on RAG...")

        emb_response = client.embeddings.create(
            model="text-embedding-3-small",
            input=query
        )

        embedding_query = emb_response.data[0].embedding

        # Vectorial search on RAG unsing LanceDB
        results = (
            table.search(embedding_query)
                 .limit(top_k)
                 .to_pandas()
        )

        print(f"✅ {len(results)} results found on RAG.")
        return results["text"].tolist()

    except Exception as e:
        print(f"Erro searching on RAG: {e}")
        return [f"Erro searching on RAG: {e}"]

In [18]:
def reasoning_process(question, search_results, finance_data, rag_context):
    steps = [
        "1️⃣ Analyze the legal and regulatory context found in the web search.",
        "2️⃣ Examine Nvidia's financial and market information.",
        "3️⃣ Integrate information from the annual report (RAG).",
        "4️⃣ Generate a conclusion with logical explanation in natural language."
    ]

    full_prompt = f"""
        You are a financial and legal analyst.
        Carefully follow the reasoning process below (Chain of Thought):

        Steps:
        {chr(10).join(steps)}

        ---
        🔍 Search results:
        {search_results}

        💹 Financial data:
        {finance_data}

        📚 Context extracted from the annual report (RAG):
        {rag_context}

        ---
        Question: {question}

        Answer in English, explaining your reasoning in a clear and structured way.
    """

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": full_prompt}],
        temperature=0.6
    )

    return response.choices[0].message.content.strip()

In [ ]:
if __name__ == "__main__":
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    db = lancedb.connect("rag/lancedb")
    table = db.open_table("vectordb")
    question = "Is there any relation between legal aspects and the financial data of Nvidia?"

    web_agent = SearchAgent()
    fin_agent = FinancialAgent()

    search_results = web_agent.search("Legal Aspects of Nvidia 2024") # It could call a LLM to format the output data.
    financial_data = fin_agent.collect_data()
    rag_context = rag_search("Legal and financial regulation of Nvidia 2024", table, client)  # It could be a dynamic system
    rag_context = " - ".join(rag_context)

    print("\n🧠 Starting reasoning process...\n")
    finan_answer = reasoning_process(question, search_results, financial_data, rag_context)

    print("✅ Final Answer:\n")
    print(finan_answer)

Generating embedding from query and searching on RAG...
✅ 3 results found on RAG.

🧠 Starting reasoning process...

✅ Final Answer:

To determine the relationship between legal aspects and the financial data of Nvidia, let's follow the outlined steps:

1️⃣ **Analyze the Legal and Regulatory Context:**
   - Nvidia is actively increasing its lobbying efforts in Washington, D.C., particularly as American lawmakers discuss regulations around artificial intelligence (AI). This indicates a proactive approach in navigating potential regulatory changes that could impact their AI-driven business model.
   - The annual report highlights various regulatory compliance areas that Nvidia must navigate, such as intellectual property, taxes, import/export requirements, anti-corruption, and more. These regulations can potentially increase operational costs or affect Nvidia's competitive position.

2️⃣ **Examine Nvidia's Financial and Market Information:**
   - Nvidia operates in the technology sector, 

In [25]:
for res in search_results:
    print(res)

{'title': 'Nvidia- Wikipedia', 'href': 'https://en.wikipedia.org/wiki/Nvidia', 'body': 'In January2024, Forbes reported thatNvidiahas increased its lobbying presence in Washington, D.C. as American lawmakers consider proposals to regulate artificial intelligence.'}
{'title': 'World Leader in Artificial Intelligence Computing |NVIDIA', 'href': 'https://www.nvidia.com/en-us/', 'body': 'NVIDIAinvents the GPU and drives advances in AI, HPC, gaming, creative design, autonomous vehicles, and robotics.'}
{'title': 'Способы скачать драйверыNvidiaв случае блокировки... / Хабр', 'href': 'https://habr.com/ru/news/853000/', 'body': 'Вечером 23 октября2024года пользователи из РФ столкнулись с блокировкой по геолокации (Access Forbidden. This request has been blocked by Edgecast WAF) возможности скачать драйверыNvidiaс основного сайта производителя видеокарт.'}


In [28]:
print(financial_data)

{'company': 'NVIDIA Corporation', 'price': np.float64(172.60499572753906), 'sector': 'Technology'}


In [27]:
print(rag_context)

ct upon our capital expenditures, 
results of operations, or competitive position and we do not currently anticipate material capital expenditures for 
environmental control facilities. Compliance with existing or future governmental regulations, including, but not limited 
to, those pertaining to IP ownership and infringement, taxes, import and export requirements and tariffs, anti-corruption, 
business acquisitions, foreign exchange controls and cash repatriation restrictions, data privacy requirements, 
competition and antitrust, advertising, employment, product regulations, cybersecurity, environmental, health and safety 
requirements, the responsible use of AI, climate change, cryptocurrency, and consumer laws, could increase our costs, 
impact our competitive position, and otherwise may have a material adverse impact on our business, financial condition 
and results of operations in subsequent periods. Refer to “Item 1A. Risk Factors” for a discussion of these potential 
impacts.